In [32]:
import pandas as pd
import ast
from collections import Counter

In [33]:
job_df = pd.read_csv("dataset/job_dataset_processed.csv")

job_df.head()

,company,job_title,job_type,experience_level,skills,skills_list,combined_text,combined_cleaned
0,SGS,clinical data analyst,Full Time,Entry-level,"['genetics', 'sas', 'computer science', 'data ...","['genetics', 'sas', 'computer science', 'data ...","clinical data analyst Entry-level ['genetics',...",clinical data analyst entry level genetics sas...
1,Ocorian,aml/cft & data analyst,Full Time,Entry-level,"['agile', 'security', 'data management', 'fina...","['agile', 'security', 'data management', 'fina...","aml/cft & data analyst Entry-level ['agile', '...",aml cft data analyst entry level agile securit...
2,Cricut,machine learning engineer,Full Time,No Experience,"['agile', 'deep learning', 'architecture', 'aw...","['agile', 'deep learning', 'architecture', 'aw...",machine learning engineer No Experience ['agil...,machine learning engineer no experience agile ...
3,Bosch Group,application developer & data analyst,Full Time,Entry-level,"['power bi', 'oracle', 'rd', 'industrial', 'en...","['power bi', 'oracle', 'rd', 'industrial', 'en...",application developer & data analyst Entry-lev...,application developer data analyst entry level...
4,Publicis Groupe,data engineer full time (public sector) usa,Full Time,Mid-level,"['data pipelines', 'azure', 'aws', 'computer s...","['data pipelines', 'azure', 'aws', 'computer s...",data engineer full time (public sector) usa Mi...,data engineer full time public sector usa mid ...


In [34]:
def parse_skills(skill_text):
    try:
        return ast.literal_eval(skill_text)
    except:
        return []

job_df["skills_list"] = job_df["skills_list"].apply(parse_skills)

In [35]:
#mappping per job
job_skill_mapping = {}

for job in job_df["job_title"].unique():
    subset = job_df[job_df["job_title"] == job]

    all_skills = []

    for skills in subset["skills_list"]:
        all_skills.extend(skills)

    top_skills = Counter(
        [skill.lower() for skill in all_skills]
    ).most_common(10)

    job_skill_mapping[job] = [skill[0] for skill in top_skills]

In [36]:
# Function to search for jobs based on keyword
def search_job(keyword):
    keyword = keyword.lower()
    return [
        job for job in job_skill_mapping.keys()
        if keyword in job.lower()
    ]

In [37]:
list(job_skill_mapping.keys())[:10]

['clinical data analyst',
 'aml/cft & data analyst',
 'machine learning engineer',
 'application developer & data analyst',
 'data engineer full time (public sector) usa',
 'sr staff data scientist - atg',
 'vendor management and data quality lead',
 'intern (business intelligence service support)',
 'summer 2023 data engineering intern',
 'principal cloud data engineer (prisma access)']

In [38]:
job_skill_mapping["clinical data analyst"]

['genetics',
 'sas',
 'computer science',
 'data quality',
 'statistics',
 'mathematics']

In [39]:
# Function to get skill gap based on job keyword
def get_skill_gap_by_keyword(user_skills, job_keyword, top_n=10):
    user_skills = [skill.lower().strip() for skill in user_skills]
    job_keyword = job_keyword.lower()

    matched_jobs = [
        job for job in job_skill_mapping.keys()
        if job_keyword in job.lower()
    ]

    all_required_skills = []

    for job in matched_jobs:
        all_required_skills.extend(job_skill_mapping[job])

    top_required_skills = [
        skill for skill, count in Counter(all_required_skills).most_common(top_n)
    ]

    missing_skills = [
        skill for skill in top_required_skills
        if skill not in user_skills
    ]

    return {
        "job_keyword": job_keyword,
        "matched_jobs_count": len(matched_jobs),
        "user_skills": user_skills,
        "required_skills": top_required_skills,
        "missing_skills": missing_skills
    }

In [40]:
def get_skill_gap_by_keyword(user_skills, job_keyword, top_n=10):
    user_skills = [skill.lower().strip() for skill in user_skills]
    job_keyword = job_keyword.lower()

    matched_jobs = [
        job for job in job_skill_mapping.keys()
        if job_keyword in job.lower()
    ]

    # jaga-jaga kalau keyword tidak ada di dataset
    if len(matched_jobs) == 0:
        return {
            "job_keyword": job_keyword,
            "matched_jobs_count": 0,
            "user_skills": user_skills,
            "required_skills": [],
            "missing_skills": [],
            "message": "Job keyword tidak ditemukan dalam dataset."
        }

    all_required_skills = []

    for job in matched_jobs:
        all_required_skills.extend(job_skill_mapping[job])

    top_required_skills = [
        skill for skill, count in Counter(all_required_skills).most_common(top_n)
    ]

    missing_skills = [
        skill for skill in top_required_skills
        if skill not in user_skills
    ]

    return {
        "job_keyword": job_keyword,
        "matched_jobs_count": len(matched_jobs),
        "user_skills": user_skills,
        "required_skills": top_required_skills,
        "missing_skills": missing_skills
    }

In [41]:
# test yaak
user_skills = ["python", "sql", "excel"]

result = get_skill_gap_by_keyword(
    user_skills,
    "data analyst"
)

result

{'job_keyword': 'data analyst',
 'matched_jobs_count': 281,
 'user_skills': ['python', 'sql', 'excel'],
 'required_skills': ['data analysis',
  'excel',
  'engineering',
  'computer science',
  'data visualization',
  'finance',
  'business intelligence',
  'power bi',
  'data analytics',
  'python'],
 'missing_skills': ['data analysis',
  'engineering',
  'computer science',
  'data visualization',
  'finance',
  'business intelligence',
  'power bi',
  'data analytics']}

In [42]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(job_skill_mapping, "models/job_skill_mapping.pkl")

['models/job_skill_mapping.pkl']

## Kesimpulan Skill Gap Recommendation

Fungsi skill gap berhasil dibuat dengan membandingkan skill pengguna dengan skill yang umum dibutuhkan pada job title tertentu. Dataset job posting digunakan sebagai knowledge base untuk mengambil required skills berdasarkan keyword pekerjaan, sehingga sistem dapat memberikan rekomendasi skill yang perlu dipelajari pengguna.